# Stap 4 - hoekmeting en PID
Upload de stap 4-sketch. De hoektest gebruikt dezelfde uitlezing als stap 3. PID stuurt PWM naar de motor op basis van doelhoek minus werkhoek.


In [ ]:
from closed_loop.meting import test_hoek, voer_pid_stap4_uit
from closed_loop.figuren import plot_pid_meting
import matplotlib.pyplot as plt
com_poort = "COM4"  # Vul de poort van jouw Arduino in.


### Ruwe hoek meten

Laat de motorvoeding uit. Geef meetduur en delta_t in seconden op. De uitvoer toont Arduino-tijd vanaf het eerste meetpunt en de ruwe sensorhoek. PWM is nul. De meting wordt als CSV opgeslagen in data/. Noteer de hoek in de nulstand; de correctie voegen we in de volgende stap toe.


In [ ]:
hoekmetingen = test_hoek(com_poort=com_poort, meetduur=1.0, delta_t=0.1)


### 2. PID instellen en meten

Begin met ki = 0 en kd = 0. Vul de gemeten beginhoek in. De werkhoek is sensorhoek - beginhoek (omgerekend naar -180..180). Controleer dat de werkhoek toeneemt als de motor de arm omhoog beweegt. De PID berekent de PWM; max_pwm begrenst deze van 0 tot en met 255 (0 houdt de motor uit). Doelhoek mag van -180 tot 180 graden zijn, gemeten vanaf stilstand. Bij beginhoek 89 en sensorhoek 89 is de werkhoek 0; bij sensorhoek 0 is de werkhoek 89 graden, ongeveer 90. Een doelhoek van 90 betekent een verplaatsing van 90 graden vanaf stilstand, niet een ruwe sensorwaarde van 90. Met toon_metingen kies je of de meetpunten live worden geprint. Fixed stopt na meetduur; continuous loopt tot je de cel onderbreekt. Delta T bepaalt het meet- en regelinterval.


In [ ]:
# Stap 4: upload Stap_4_PID.ino.
beginhoek = 115.4  # Vervang door de gemeten sensorhoek in de nulstand.
doelhoek = 90.0    # graden ten opzichte van de ingestelde nulstand
kp = 3
ki = 3
kd = 3
max_pwm = 255     # bovengrens; kies een lagere waarde indien nodig
meetduur = 10.0   # seconden; stoptijd in fixed mode
delta_t = 0.02    # seconden (0.02-1); frequentie = 1 / delta_t Hz
mode = "fixed"   # "fixed" of "continuous" (stop met cel onderbreken)

toon_metingen = True  # False: geen meetpunten printen

metingen, csv_pad = voer_pid_stap4_uit(
    com_poort=com_poort, beginhoek=beginhoek,
    doelhoek=doelhoek,
    kp=kp, ki=ki, kd=kd,
    max_pwm=max_pwm, toon_metingen=toon_metingen,
    meetduur=meetduur, delta_t=delta_t, mode=mode
)

### 3. Resultaat bekijken en CSV inlezen

De PID-meting wordt opgeslagen in een CSV met alle instellingen bovenaan (regels die met `#` beginnen), gevolgd door `tijd_s,hoek_deg,pwm`. Er wordt geen apart JSON-bestand meer aangemaakt. Tijden in de instellingen met `_ms` zijn in milliseconden; `tijd_s` in de meetdata is in seconden.

De cel hieronder leest de laatst opgeslagen PID-meting in. Vervang `csv_pad` door een bestandspad om een andere meting te openen. Voor een veiligheidsproef gebruik je `csv_pad_veiligheid`. Deze uitleescel is bedoeld voor het nieuwe CSV-formaat.


In [ ]:
import csv
import pandas as pd

bestand = csv_pad  # Of bijvoorbeeld "data/pid_20261027_....csv"

# Lees de instellingen uit de commentaarheader.
with open(bestand, encoding="utf-8", newline="") as f:
    instellingen = {}
    for rij in csv.reader(f):
        if not rij or not rij[0].startswith("# "):
            break
        instellingen[rij[0][2:]] = rij[1]

# Pandas slaat de commentaarheader over en leest de meetkolommen.
df = pd.read_csv(bestand, comment="#")
display(pd.Series(instellingen, name="Instelling").to_frame())
display(df.head())


### 4. De maximale hoek aanpassen

Gebruik dezelfde functie en dezelfde geuploade sketch. Tot nu toe hoefde je geen `max_hoek` mee te geven: de functie gebruikt standaard 180 graden. Geef nu `max_hoek=120` mee. Bij overschrijding van de ingestelde werkhoek stopt de Arduino de motor en de meting. De doelhoek moet binnen de ingestelde grens liggen.


In [ ]:
# Vervolg: dezelfde sketch en functie, nu met een lagere stopgrens.
beginhoek = 115.4  # Vervang door de gemeten sensorhoek in de nulstand.
doelhoek = 90.0    # graden ten opzichte van de ingestelde nulstand
kp = 4
ki = 3
kd = 0
max_pwm = 255     # bovengrens; kies een lagere waarde indien nodig
meetduur = 10.0   # seconden; stoptijd in fixed mode
delta_t = 0.02    # seconden (0.02-1); frequentie = 1 / delta_t Hz
mode = "fixed"   # "fixed" of "continuous" (stop met cel onderbreken)

toon_metingen = True 
max_hoek = 100.0  # Stopgrens voor de werkhoek; kies passend bij de opstelling.

metingen_veiligheid, csv_pad_veiligheid = voer_pid_stap4_uit(
    com_poort=com_poort, beginhoek=beginhoek,
    doelhoek=doelhoek, max_hoek=max_hoek,
    kp=kp, ki=ki, kd=kd,
    max_pwm=max_pwm, toon_metingen=toon_metingen,
    meetduur=meetduur, delta_t=delta_t, mode=mode,
)
